# Modeling Preprocessing
Chuẩn bị `label_df_final` thành tập dữ liệu sẵn sàng cho training: tạo nhãn dự báo (forecast label) để tránh leakage, loại nhãn không xác định, chọn feature, xử lý NaN còn lại, và chia train/val/test theo bệnh nhân (patient-level, stratified).

**Bối cảnh:**
- Bài toán được đóng khung lại thành **early-warning / forecasting**: dự đoán `risk_level` tại thời điểm *t + FORECAST_HORIZON_MIN phút tới*, dựa trên toàn bộ feature (kể cả SPO2/PULSE thô) tại thời điểm *t*. Nhờ đó SPO2/PULSE hiện tại không còn là leakage (chúng không trùng thời điểm với nhãn).
- Giai đoạn "cold-start" (chưa đủ 30 phút lịch sử) sẽ được xử lý ở tầng ứng dụng bằng rule-based NEWS2 score có sẵn (`spo2_score`, `pulse_score`), không phải việc của notebook này.
- Do phân bố `Critical`/`Alert` lệch mạnh theo bệnh nhân (top 3/47 bệnh nhân chiếm 61% số dòng Critical), split phải thực hiện theo `patient_id` (group) **và** stratify theo mức risk cao nhất mà bệnh nhân từng đạt.

## 5.0 Import & cấu hình

In [1]:
import pandas as pd
import numpy as np
import json
import os

CHECKPOINT_DIR = "../data/checkpoints"
LABEL_DF_PATH = f"{CHECKPOINT_DIR}/label_df_final.parquet"

GROUP_KEYS = ["patient_id", "record_path"]
FORECAST_HORIZON_MIN = 30   # dự báo risk_level sẽ ra sao sau 30 phút nữa
RANDOM_STATE = 42

RISK_ORDER = {"Normal": 0, "Watch": 1, "Alert": 2, "Critical": 3}

## 5.1 Load checkpoint & tổng quan
Đọc `label_df_final.parquet` từ bước data-processing, sắp xếp lại theo (patient_id, record_path, timestamp) để đảm bảo các phép `shift` ở bước 5.2 đúng thứ tự thời gian.

In [2]:
label_df = pd.read_parquet(LABEL_DF_PATH)
label_df = label_df.sort_values(GROUP_KEYS + ["timestamp"]).reset_index(drop=True)

print(f"label_df: {label_df.shape[0]:,} dòng x {label_df.shape[1]} cột, "
      f"{label_df['patient_id'].nunique()} bệnh nhân")
label_df["risk_level"].value_counts(dropna=False)

label_df: 89,543 dòng x 46 cột, 47 bệnh nhân


risk_level
Watch       39229
Normal      34471
Alert       10139
Unknown      4239
Critical     1465
Name: count, dtype: int64

## 5.2 Tạo nhãn dự báo (forecast label)
Với mỗi dòng tại thời điểm t, nhãn train sẽ là `risk_level` tại thời điểm **t + FORECAST_HORIZON_MIN phút** (shift theo nhóm patient_id/record_path để không lấy nhầm dữ liệu bệnh nhân khác). Các dòng ở cuối mỗi record không đủ tương lai sẽ tự nhiên thành NaN và bị loại ở bước 5.3.

In [3]:
label_df["risk_level_future"] = (
    label_df.groupby(GROUP_KEYS)["risk_level"].shift(-FORECAST_HORIZON_MIN)
)

n_tail_nan = label_df["risk_level_future"].isna().sum()
print(f"Số dòng cuối record không đủ tương lai để gán nhãn (sẽ bị loại): {n_tail_nan:,}")
label_df["risk_level_future"].value_counts(dropna=False)

Số dòng cuối record không đủ tương lai để gán nhãn (sẽ bị loại): 1,410


risk_level_future
Watch       38582
Normal      34171
Alert        9871
Unknown      4119
NaN          1410
Critical     1390
Name: count, dtype: int64

## 5.3 Loại bỏ nhãn "Unknown" và dòng thiếu nhãn tương lai
Loại các dòng mà:
- `risk_level` hiện tại là "Unknown" (SPO2/PULSE tại t vẫn NaN sau fill ở bước 2), hoặc
- `risk_level_future` là NaN (cuối record, không đủ 30 phút tương lai), hoặc
- `risk_level_future` là "Unknown" (tại t+30' cũng đang mất tín hiệu).

Đây là bước tiền xử lý dành riêng cho modeling (không sửa lại `label_df_final` gốc).

In [4]:
before = len(label_df)

model_df = label_df[
    (label_df["risk_level"] != "Unknown")
    & (label_df["risk_level_future"].notna())
    & (label_df["risk_level_future"] != "Unknown")
].copy()

after = len(model_df)
print(f"Trước lọc: {before:,} dòng -> Sau lọc: {after:,} dòng ({after/before*100:.1f}% giữ lại)")
model_df["risk_level_future"].value_counts(normalize=True).mul(100).round(2)

Trước lọc: 89,543 dòng -> Sau lọc: 82,503 dòng (92.1% giữ lại)


risk_level_future
Watch       45.91
Normal      40.87
Alert       11.63
Critical     1.59
Name: proportion, dtype: float64

## 5.4 Chọn feature — loại cột định danh & cột không nên dùng
Loại khỏi feature set:
- `patient_id`, `record_path`, `timestamp`: định danh / thời gian (patient_id giữ lại riêng để group split ở bước 5.7).
- `source_fs_hz`, `dropout_ratio_full_record`: giá trị gần như hằng số theo từng record → dễ trở thành "shortcut" giúp model nhận diện bệnh nhân thay vì học pattern sinh lý thật.
- `risk_level_future`: chính là target.

Giữ lại `SPO2`, `PULSE` thô, các feature rolling/slope/desaturation, và `spo2_score`/`pulse_score`/`news2_partial_score`/`risk_level` (hiện tại) — đều là thông tin có tại thời điểm t, hợp lệ để dự đoán trạng thái tại t+30'.

In [5]:
EXCLUDE_COLS = [
    "patient_id", "record_path", "timestamp",
    "source_fs_hz", "dropout_ratio_full_record",
    "risk_level_future",
]
FEATURE_COLS = [c for c in model_df.columns if c not in EXCLUDE_COLS]
TARGET_COL = "risk_level_future"

print(f"Số feature sử dụng: {len(FEATURE_COLS)}")
print(FEATURE_COLS)

Số feature sử dụng: 41
['SPO2', 'PULSE', 'SPO2_mean_5m', 'SPO2_std_5m', 'SPO2_min_5m', 'SPO2_max_5m', 'SPO2_mean_15m', 'SPO2_std_15m', 'SPO2_min_15m', 'SPO2_max_15m', 'SPO2_mean_30m', 'SPO2_std_30m', 'SPO2_min_30m', 'SPO2_max_30m', 'PULSE_mean_5m', 'PULSE_std_5m', 'PULSE_min_5m', 'PULSE_max_5m', 'PULSE_mean_15m', 'PULSE_std_15m', 'PULSE_min_15m', 'PULSE_max_15m', 'PULSE_mean_30m', 'PULSE_std_30m', 'PULSE_min_30m', 'PULSE_max_30m', 'SPO2_slope_5m', 'SPO2_slope_15m', 'SPO2_slope_30m', 'PULSE_slope_5m', 'PULSE_slope_15m', 'PULSE_slope_30m', 'spo2_drop_from_baseline', 'is_desat_event', 'desat_index_5m', 'desat_index_15m', 'desat_index_30m', 'spo2_score', 'pulse_score', 'news2_partial_score', 'risk_level']


## 5.5 Kiểm tra NaN còn lại trong feature
Các NaN còn sót chủ yếu ở feature rolling (đầu mỗi record chưa đủ window, hoặc gap dài hơn 5' không được fill ở bước 2). Chính sách: **giữ nguyên NaN**, không impute ở bước tiền xử lý này — vì:
- Model tree-based (LightGBM/XGBoost) xử lý NaN native, thường tốt hơn impute cứng.
- Impute (nếu cần cho model tuyến tính/khoảng cách) nên làm trong pipeline training, fit trên tập train để tránh leakage thống kê từ val/test.

In [6]:
na_ratio = model_df[FEATURE_COLS].isna().mean().sort_values(ascending=False)
na_ratio = na_ratio[na_ratio > 0]
print(f"Số feature còn NaN: {len(na_ratio)} / {len(FEATURE_COLS)}")
na_ratio.mul(100).round(2)

Số feature còn NaN: 31 / 41


SPO2_mean_30m              1.44
SPO2_max_30m               1.44
SPO2_min_30m               1.44
SPO2_std_30m               1.44
SPO2_slope_30m             1.44
PULSE_mean_30m             1.43
PULSE_std_30m              1.43
PULSE_min_30m              1.43
PULSE_max_30m              1.43
PULSE_slope_30m            1.43
SPO2_std_15m               0.69
SPO2_slope_15m             0.69
SPO2_max_15m               0.69
SPO2_min_15m               0.69
SPO2_mean_15m              0.69
PULSE_std_15m              0.67
PULSE_min_15m              0.67
PULSE_slope_15m            0.67
PULSE_mean_15m             0.67
PULSE_max_15m              0.67
spo2_drop_from_baseline    0.15
SPO2_mean_5m               0.13
SPO2_std_5m                0.13
SPO2_min_5m                0.13
SPO2_max_5m                0.13
SPO2_slope_5m              0.13
PULSE_mean_5m              0.12
PULSE_max_5m               0.12
PULSE_min_5m               0.12
PULSE_std_5m               0.12
PULSE_slope_5m             0.12
dtype: f

## 5.6 Gán "dominant risk" theo bệnh nhân
Với mỗi bệnh nhân, lấy mức risk cao nhất từng đạt được (theo `risk_level_future`) làm biến stratify khi chia patient vào train/val/test — đảm bảo mỗi tập đều có bệnh nhân từng đạt Critical/Alert.

In [7]:
patient_dominant = (
    model_df.groupby("patient_id")["risk_level_future"]
    .apply(lambda s: max(s, key=lambda x: RISK_ORDER[x]))
    .rename("dominant_risk")
    .reset_index()
)

print("Phân bố dominant_risk theo bệnh nhân:")
patient_dominant["dominant_risk"].value_counts()

Phân bố dominant_risk theo bệnh nhân:


dominant_risk
Critical    31
Alert       14
Watch        2
Name: count, dtype: int64

## 5.7 Tách test cố định + gán fold cho dev_pool (group k-fold)
Thay vì 1 val cố định (dễ bị lệch nếu 1 bệnh nhân "khủng" rơi trúng val — như đã phát hiện ở lần đánh giá trước), chiến lược mới:

1. **Tách trước `test`** (~15% bệnh nhân, stratify theo `dominant_risk`) — tập này giữ cố định, chỉ dùng 1 lần duy nhất ở cuối để đánh giá model tốt nhất, không dùng để tune.
2. **Phần còn lại → `dev_pool`**, gán mỗi bệnh nhân 1 `fold_id` (0..N_FOLDS-1) theo group k-fold, rải đều theo `dominant_risk` (round-robin trong từng nhóm risk) — để không có 1 fold nào ôm gần hết bệnh nhân Critical.

In [12]:
TEST_SIZE = 0.15
MIN_GROUP_FOR_SPLIT = 3
N_FOLDS = 5

def split_test_holdout(patient_dominant, test_size, random_state, min_group_for_split):
    rng = np.random.RandomState(random_state)
    dev_ids, test_ids, skipped_strata = [], [], []
    for stratum, grp in patient_dominant.groupby("dominant_risk"):
        ids = grp["patient_id"].tolist()
        rng.shuffle(ids)
        n = len(ids)
        if n < min_group_for_split:
            dev_ids += ids
            skipped_strata.append((stratum, n))
            continue
        n_test = max(1, round(n * test_size))
        test_ids += ids[:n_test]
        dev_ids += ids[n_test:]
    return sorted(dev_ids), sorted(test_ids), skipped_strata

def assign_folds(dev_ids, patient_dominant, n_folds, random_state):
    """Round-robin gán fold_id trong từng nhóm dominant_risk, để mỗi fold có đủ
    đại diện các mức risk thay vì dồn hết 1 nhóm hiếm vào 1 fold."""
    rng = np.random.RandomState(random_state)
    dev_dominant = patient_dominant[patient_dominant["patient_id"].isin(dev_ids)]
    fold_map = {}
    for stratum, grp in dev_dominant.groupby("dominant_risk"):
        ids = grp["patient_id"].tolist()
        rng.shuffle(ids)
        for i, pid in enumerate(ids):
            fold_map[pid] = i % n_folds
    return fold_map

dev_ids, test_ids, skipped_strata = split_test_holdout(
    patient_dominant, TEST_SIZE, RANDOM_STATE, MIN_GROUP_FOR_SPLIT
)
if skipped_strata:
    print("[LƯU Ý] Nhóm dominant_risk quá ít bệnh nhân để tách test, giữ hết trong dev_pool:")
    for stratum, n in skipped_strata:
        print(f"  - {stratum}: {n} bệnh nhân")

fold_map = assign_folds(dev_ids, patient_dominant, N_FOLDS, RANDOM_STATE)

print(f"\nSố bệnh nhân -> dev_pool: {len(dev_ids)}, test (giữ cố định): {len(test_ids)}")

test_df = model_df[model_df["patient_id"].isin(test_ids)].reset_index(drop=True)
dev_df = (
    model_df[model_df["patient_id"].isin(dev_ids)]
    .merge(pd.Series(fold_map, name="fold_id").rename_axis("patient_id").reset_index(),
           on="patient_id", how="left")
    .reset_index(drop=True)
)

print(f"Số dòng -> dev_pool: {len(dev_df):,}, test: {len(test_df):,}")

[LƯU Ý] Nhóm dominant_risk quá ít bệnh nhân để tách test, giữ hết trong dev_pool:
  - Watch: 2 bệnh nhân

Số bệnh nhân -> dev_pool: 40, test (giữ cố định): 7
Số dòng -> dev_pool: 69,409, test: 13,094


## 5.8 Kiểm tra phân bố nhãn: test + từng fold trong dev_pool
Thay vì so sánh train/val/test, giờ kiểm tra:
- Phân bố nhãn của `test` (tập giữ cố định).
- Số bệnh nhân / số dòng / số dòng Critical theo từng `fold_id` trong `dev_pool` — để biết trước fold nào "nặng" Critical hơn (do bệnh nhân đóng góp nhiều dòng Critical nhất rơi vào fold nào), tham khảo khi đọc kết quả cross-validation ở bước train.

In [14]:
print("Phân bố nhãn tập test (%):")
display(test_df["risk_level_future"].value_counts(normalize=True).mul(100).round(2)
        .reindex(["Normal", "Watch", "Alert", "Critical"]))

print("\nSố bệnh nhân theo từng fold (dev_pool):")
display(dev_df.groupby("fold_id")["patient_id"].nunique())

print("\nSố dòng theo từng fold:")
display(dev_df.groupby("fold_id").size())

print("\nSố dòng nhãn Critical theo từng fold (fold nào có bệnh nhân đóng góp nhiều Critical sẽ nổi rõ ở đây):")
display(dev_df[dev_df["risk_level_future"] == "Critical"].groupby("fold_id").size())

Phân bố nhãn tập test (%):


risk_level_future
Normal      13.35
Watch       70.38
Alert       15.25
Critical     1.02
Name: proportion, dtype: float64


Số bệnh nhân theo từng fold (dev_pool):


fold_id
0    10
1     9
2     7
3     7
4     7
Name: patient_id, dtype: int64


Số dòng theo từng fold:


fold_id
0    16210
1    17916
2    13861
3    11700
4     9722
dtype: int64


Số dòng nhãn Critical theo từng fold (fold nào có bệnh nhân đóng góp nhiều Critical sẽ nổi rõ ở đây):


fold_id
0     65
1    127
2     87
3    742
4    157
dtype: int64

## 5.9 Lưu checkpoint
Lưu:
- `model_df.parquet`: toàn bộ dữ liệu đã gắn nhãn dự báo, trước khi split (giữ như cũ).
- `dev_df.parquet`: toàn bộ dev_pool kèm cột `fold_id` — bước train sẽ tự lặp qua các fold (fold_id == k làm validation, còn lại làm train) chứ không đọc `train_df`/`val_df` cố định nữa.
- `test_df.parquet`: tập test cố định, chỉ dùng 1 lần cuối cùng.
- `split_manifest.json`: cấu hình (horizon, random_state, N_FOLDS, feature_cols, target_col) + `test_patient_ids` + `fold_assignment` (patient_id -> fold_id) — để bước train tái lập đúng, không tự chia lại.

In [15]:
model_df.to_parquet(f"{CHECKPOINT_DIR}/model_df.parquet", index=False)
dev_df.to_parquet(f"{CHECKPOINT_DIR}/dev_df.parquet", index=False)
test_df.to_parquet(f"{CHECKPOINT_DIR}/test_df.parquet", index=False)

split_manifest = {
    "forecast_horizon_min": FORECAST_HORIZON_MIN,
    "random_state": RANDOM_STATE,
    "test_size": TEST_SIZE,
    "n_folds": N_FOLDS,
    "feature_cols": FEATURE_COLS,
    "target_col": TARGET_COL,
    "test_patient_ids": test_ids,
    "fold_assignment": fold_map,
}
with open(f"{CHECKPOINT_DIR}/split_manifest.json", "w", encoding="utf-8") as f:
    json.dump(split_manifest, f, ensure_ascii=False, indent=2)

print("Đã lưu checkpoint vào", CHECKPOINT_DIR)
os.listdir(CHECKPOINT_DIR)

Đã lưu checkpoint vào ../data/checkpoints


['clean_df.parquet',
 'dev_df.parquet',
 'feat_df.parquet',
 'label_df_final.parquet',
 'model_df.parquet',
 'raw_df.parquet',
 'split_manifest.json',
 'test_df.parquet',
 'train_df.parquet',
 'val_df.parquet']

## 5.10 Tổng kết Bước 5
- Nhãn đã chuyển từ "nowcasting" sang "forecast t+30 phút" (`FORECAST_HORIZON_MIN`), loại bỏ leakage giữa SPO2/PULSE hiện tại và nhãn.
- Đã loại 100% nhãn "Unknown" (hiện tại & tương lai) và các dòng cuối record thiếu tương lai — giữ lại 82,503/89,543 dòng (92.1%).
- Đã loại các cột định danh/gần-hằng-số (`patient_id`, `record_path`, `timestamp`, `source_fs_hz`, `dropout_ratio_full_record`) khỏi feature set để tránh model học "tắt" theo bệnh nhân — còn lại 41 feature, trong đó `risk_level` (hiện tại) vẫn là dạng string, **cần encode ở bước train** trước khi đưa vào model tuyến tính (tree-based như LightGBM/XGBoost có thể nhận categorical trực tiếp).
- NaN trong feature rolling (tối đa ~1.44% ở window 30 phút) được giữ nguyên, để lại quyết định impute/native-NaN cho bước training.
- **Chiến lược split đã đổi so với bản đầu:** không còn train/val/test 3 phần cố định. Thay vào đó:
  - Tách trước **`test` cố định** (7 bệnh nhân, ~15%, stratify theo `dominant_risk`) — chỉ dùng 1 lần duy nhất ở cuối để đánh giá model đã chọn, không dùng để tune.
  - Phần còn lại là **`dev_pool`** (40 bệnh nhân) được gán `fold_id` (0-4) theo group k-fold, rải đều theo `dominant_risk` — ranh giới fold được chốt cứng trong `split_manifest.json` (`fold_assignment`) để bước train tái lập đúng, không tự chia lại.
  - **Lưu ý quan trọng khi đọc kết quả CV:** do chỉ có 47 bệnh nhân và 1 bệnh nhân (`p013569`) chiếm 41% tổng số dòng Critical, fold 3 hiện đang "nặng" Critical hơn hẳn các fold khác (742 dòng so với 65-157 ở 4 fold còn lại). Ở bước train, **bắt buộc phải report riêng metric từng fold** (đặc biệt recall/precision của Alert & Critical), không chỉ lấy trung bình 5 fold, để tránh hiểu nhầm model ổn định đều khi thực chất đang bị 1 fold chi phối.

**Bước tiếp theo:** `model-training` (branch `feature/06-model-training`) — encode `risk_level` hiện tại nếu cần, thử baseline (Logistic Regression + class_weight) so với tree-based (LightGBM/XGBoost, NaN-native, categorical-native), chạy group k-fold theo `fold_assignment` đã lưu, report riêng recall/precision theo từng fold cho lớp Alert & Critical, cuối cùng mới đánh giá 1 lần trên `test_df` với model đã chốt.